# 06 Modèles linéaires, régression après ACP et k plus proches voisins

Même protocole que le notebook 05 : mêmes plis par groupes, cible recalculée dans chaque apprentissage, préparation ajustée dans chaque pli, RMSE pondérée par groupe. Les résultats sont donc directement comparables à ceux des arbres.

Modèles testés :
- régression linéaire, Ridge et Lasso ;
- régression linéaire après ACP ;
- k plus proches voisins.

*Texte d'introduction à rédiger.*

## 1 Reprendre les données et le protocole du notebook 05

In [ ]:
from pathlib import Path
import json, time, warnings, hashlib, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs/06'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print('Version de scikit-learn :', sklearn_version)
pd.set_option('display.max_colwidth', None)

In [ ]:
categories = ['AC_DC', 'Electrode_pol', 'WeldType']
source = ROOT / 'outputs/02/dataset_traction.csv'
donnees = pd.read_csv(source, index_col='row_id',
    dtype={c: str for c in categories + ['WeldID', 'partition']},
    keep_default_na=False, na_values=[''], float_precision='round_trip')
schema = json.loads((ROOT / 'data/processed/welddb_metadata.json').read_text())['schema']
variables_X = schema['COMPOSITION'] + schema['PROCESS']
mesures = ['UTS_MPa', 'Elongation_pct']
X = donnees[variables_X]
Y = donnees[mesures]
groupes = donnees['groupe']
dev = donnees['partition'].eq('developpement')
test = donnees['partition'].eq('test')
assert (dev | test).all()
assert set(groupes[dev]).isdisjoint(set(groupes[test]))
X_dev, Y_dev, G_dev = X.loc[dev], Y.loc[dev], groupes.loc[dev]
print(f'{len(X_dev)} lignes de développement ; {int(test.sum())} réservées au test.')

In [ ]:
def poids_groupes(g):
    return (1 / g.map(g.value_counts())).to_numpy(dtype=float)

def calculer_cible(Y_apprentissage, groupes_apprentissage, Y_a_transformer):
    poids = poids_groupes(groupes_apprentissage)
    rangs = pd.DataFrame(index=Y_a_transformer.index)
    for mesure in mesures:
        tableau = pd.DataFrame({'valeur': Y_apprentissage[mesure].to_numpy(), 'poids': poids})
        masse = tableau.groupby('valeur')['poids'].sum()
        reference = (masse.cumsum() - masse / 2) / masse.sum()
        rangs[mesure] = np.interp(Y_a_transformer[mesure], masse.index, reference)
    return np.sqrt(rangs.prod(axis=1))

def mesurer(y_reel, prediction, g):
    poids = poids_groupes(g)
    return {
        'RMSE_groupes': np.sqrt(mean_squared_error(y_reel, prediction, sample_weight=poids)),
        'MAE_groupes': mean_absolute_error(y_reel, prediction, sample_weight=poids),
        'R2_groupes': r2_score(y_reel, prediction, sample_weight=poids),
        'RMSE_lignes': np.sqrt(mean_squared_error(y_reel, prediction))
    }

## 2 Une préparation adaptée aux modèles sensibles aux échelles

Différences avec le notebook 05 :
- les nombres sont **standardisés** (indispensable pour Ridge, Lasso, ACP et k plus proches voisins) ;
- option `indicateurs` : une colonne 0/1 par entrée numérique incomplète, qui indique si la valeur a été imputée ;
- option `n_axes` : une ACP sur les nombres standardisés, les catégories encodées restant à côté (comme dans le notebook 04, l'ACP ne porte que sur les variables numériques).

*Justification à rédiger.*

In [ ]:
def creer_preparation(X_apprentissage, indicateurs=False, n_axes=None):
    taux = X_apprentissage.notna().mean()
    gardees = taux[taux >= 0.25].index.tolist()
    numeriques = [c for c in gardees if c not in categories]
    categorielles = [c for c in gardees if c in categories]
    etapes = [('mediane', SimpleImputer(strategy='median', add_indicator=indicateurs)),
              ('standardisation', StandardScaler())]
    if n_axes is not None:
        etapes.append(('acp', PCA(n_components=min(n_axes, len(numeriques)), random_state=SEED)))
    return ColumnTransformer([
        ('nombres', Pipeline(etapes), numeriques),
        ('categories', Pipeline([
            ('manquants', SimpleImputer(strategy='constant', fill_value='manquant')),
            ('encodage', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ]), categorielles)
    ])

cache = tempfile.TemporaryDirectory(prefix='soudures_pipeline_')

def creer_pipeline(X_apprentissage, modele, preparation=None):
    return Pipeline([
        ('preparation', creer_preparation(X_apprentissage, **(preparation or {}))),
        ('modele', clone(modele))
    ], memory=cache.name)

In [ ]:
plis = []
validation = GroupKFold(n_splits=5)
for train, val in validation.split(X_dev, groups=G_dev):
    ids_train, ids_val = X_dev.index[train], X_dev.index[val]
    assert set(groupes.loc[ids_train]).isdisjoint(set(groupes.loc[ids_val]))
    plis.append({
        'X_train': X.loc[ids_train], 'X_val': X.loc[ids_val],
        'y_train': calculer_cible(Y.loc[ids_train], groupes.loc[ids_train], Y.loc[ids_train]),
        'y_val': calculer_cible(Y.loc[ids_train], groupes.loc[ids_train], Y.loc[ids_val]),
        'g_train': groupes.loc[ids_train], 'g_val': groupes.loc[ids_val]
    })

details_05 = pd.read_csv(ROOT / 'outputs/05/resultats_par_pli.csv')
reference_05 = details_05[details_05['modele'].eq('Moyenne constante')]['RMSE_groupes'].to_numpy()
print('5 plis par groupes, identiques à ceux du notebook 05.')

## 3 Boucle d'évaluation

Même boucle que le notebook 05, avec deux ajouts : les paramètres de la préparation font partie de la grille (préfixe `preparation__`), et les k plus proches voisins ne reçoivent pas de poids par groupe, car `KNeighborsRegressor.fit` n'accepte pas `sample_weight`. L'évaluation reste pondérée par groupe pour tous les modèles.

In [ ]:
details_plis = []

def tester_grille(nom, modele, grille, poids=True):
    lignes = []
    for numero, parametres in enumerate(ParameterGrid(grille)):
        params_prep = {k.removeprefix('preparation__'): v for k, v in parametres.items()
                       if k.startswith('preparation__')}
        params_modele = {k: v for k, v in parametres.items() if not k.startswith('preparation__')}
        erreurs = []
        debut = time.perf_counter()
        for pli, d in enumerate(plis, start=1):
            pipeline = creer_pipeline(d['X_train'], modele, params_prep)
            pipeline.set_params(**{'modele__' + k: v for k, v in params_modele.items()})
            options = {'modele__sample_weight': poids_groupes(d['g_train'])} if poids else {}
            pipeline.fit(d['X_train'], d['y_train'], **options)
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', message='Found unknown categories.*', category=UserWarning)
                prediction = pipeline.predict(d['X_val'])
            scores = mesurer(d['y_val'], prediction, d['g_val'])
            scores['RMSE_apprentissage'] = mesurer(
                d['y_train'], pipeline.predict(d['X_train']), d['g_train'])['RMSE_groupes']
            erreurs.append(scores)
            details_plis.append({'modele': nom, 'configuration': numero, 'pli': pli, **scores})
        tableau = pd.DataFrame(erreurs)
        lignes.append({'modele': nom, 'configuration': numero, 'parametres': parametres,
                       **tableau.mean().to_dict(),
                       'ecart_type_RMSE': tableau['RMSE_groupes'].std(ddof=1),
                       'temps_secondes': time.perf_counter() - debut})
    resultats = pd.DataFrame(lignes).sort_values('RMSE_groupes', kind='stable')
    print(f'{nom} : {len(resultats)} configurations sur 5 plis.')
    display(resultats[['parametres', 'RMSE_groupes', 'ecart_type_RMSE',
                      'RMSE_apprentissage']].head(5).round(4))
    return resultats

reference = tester_grille('Moyenne constante', DummyRegressor(strategy='mean'), {})
np.testing.assert_allclose(
    pd.DataFrame(details_plis)['RMSE_groupes'].to_numpy(), reference_05, atol=1e-12)

## 4 Régression linéaire

Trois variantes :
- **moindres carrés** sans pénalité ;
- **Ridge** : pénalité sur la somme des carrés des coefficients ;
- **Lasso** : pénalité sur la somme des valeurs absolues, qui peut annuler des coefficients.

La force de pénalité `alpha` est choisie en validation. Nous testons aussi l'ajout des indicateurs de valeurs imputées.

In [ ]:
lineaire = tester_grille('Régression linéaire', LinearRegression(),
                         {'preparation__indicateurs': [False, True]})
ridge = tester_grille('Ridge', Ridge(),
                      {'alpha': list(np.logspace(-3, 3, 13)),
                       'preparation__indicateurs': [False, True]})
lasso = tester_grille('Lasso', Lasso(max_iter=50000),
                      {'alpha': list(np.logspace(-5, -1, 13)),
                       'preparation__indicateurs': [False, True]})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for nom, tableau in [('Ridge', ridge), ('Lasso', lasso)]:
    for indic, style in [(False, '-'), (True, '--')]:
        sel = tableau[tableau['parametres'].apply(lambda p: p['preparation__indicateurs'] == indic)].copy()
        sel['alpha'] = sel['parametres'].apply(lambda p: p['alpha'])
        sel = sel.sort_values('alpha')
        ax.plot(sel['alpha'], sel['RMSE_groupes'], style, marker='o', markersize=3,
                label=f'{nom}, indicateurs={indic}')
ax.axhline(reference['RMSE_groupes'].iloc[0], color='grey', linestyle=':', label='Moyenne constante')
ax.set(xscale='log', xlabel='alpha', ylabel='RMSE de validation par groupes',
       title='Effet de la pénalité')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'penalite_lineaire.png', dpi=150); plt.show()

### Lire les coefficients du meilleur modèle linéaire

Nous réentraînons le meilleur modèle linéaire pénalisé sur tout le développement. Les entrées numériques étant standardisées, un coefficient se lit comme la variation du score prédit pour un écart-type de la variable, **les autres entrées étant fixées**. Ce n'est pas un effet causal, et des variables corrélées peuvent se partager ou s'échanger leurs coefficients.

In [ ]:
y_dev_final = calculer_cible(Y_dev, G_dev, Y_dev)
np.testing.assert_allclose(y_dev_final, donnees.loc[dev, 'score_compromis'], atol=1e-12)

meilleur_penalise = min([ridge, lasso], key=lambda t: t['RMSE_groupes'].iloc[0])
ligne = meilleur_penalise.iloc[0]
modele_lin = {'Ridge': Ridge(), 'Lasso': Lasso(max_iter=50000)}[ligne['modele']]
params = ligne['parametres']
pipeline_lin = creer_pipeline(
    X_dev, modele_lin.set_params(alpha=params['alpha']),
    {'indicateurs': params['preparation__indicateurs']})
pipeline_lin.fit(X_dev, y_dev_final, modele__sample_weight=poids_groupes(G_dev))
coefficients = pd.Series(pipeline_lin.named_steps['modele'].coef_,
                         index=pipeline_lin.named_steps['preparation'].get_feature_names_out())
coefficients = coefficients[coefficients.abs().sort_values(ascending=False).index]
print(ligne['modele'], params)
print(f'{(coefficients != 0).sum()} coefficients non nuls sur {len(coefficients)}.')

fig, ax = plt.subplots(figsize=(7, 7))
haut = coefficients.head(20)[::-1]
ax.barh(haut.index, haut.to_numpy(), color=np.where(haut > 0, 'tab:blue', 'tab:red'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set(xlabel='Coefficient', title=f'20 plus grands coefficients ({ligne["modele"]})')
fig.tight_layout(); fig.savefig(OUT / 'coefficients_lineaire.png', dpi=150); plt.show()

## 5 Régression linéaire après ACP

L'ACP est ajustée **dans chaque pli**, sur les nombres standardisés de l'apprentissage. Le modèle linéaire utilise les `n_axes` premières composantes, plus les catégories encodées. Le nombre d'axes est choisi en validation.

Rappel du notebook 04 : 11 axes pour 80 % de la variance, 14 pour 90 %. Une forte variance expliquée ne garantit pas une bonne prédiction, car l'ACP ne regarde pas la cible.

In [ ]:
axes_testes = [1, 2, 3, 4, 6, 8, 10, 11, 12, 14, 16, 18, 20, 22]
acp_lineaire = tester_grille('Linéaire après ACP', LinearRegression(),
                             {'preparation__n_axes': axes_testes})

In [ ]:
courbe = acp_lineaire.copy()
courbe['n_axes'] = courbe['parametres'].apply(lambda p: p['preparation__n_axes'])
courbe = courbe.sort_values('n_axes')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(courbe['n_axes'], courbe['RMSE_apprentissage'], marker='o', label='Apprentissage')
ax.plot(courbe['n_axes'], courbe['RMSE_groupes'], marker='o', label='Validation')
ax.axhline(lineaire['RMSE_groupes'].min(), color='grey', linestyle='--', label='Linéaire sans ACP')
ax.axvline(11, color='lightgrey', linestyle=':'); ax.axvline(14, color='lightgrey', linestyle=':')
ax.set(xlabel='Nombre d’axes conservés', ylabel='RMSE par groupes',
       title='Régression linéaire après ACP')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'acp_nombre_axes.png', dpi=150); plt.show()

## 6 k plus proches voisins

La prédiction est la moyenne des scores des `k` soudures d'apprentissage les plus proches dans l'espace des entrées préparées. Nous testons :
- `k` de 1 à 50 ;
- une moyenne simple ou pondérée par l'inverse de la distance ;
- la distance de Manhattan (`p=1`) ou euclidienne (`p=2`) ;
- avec ou sans ACP préalable.

Les variables doivent être standardisées : sinon le courant ou la température domineraient la distance.

In [ ]:
knn = tester_grille('k plus proches voisins', KNeighborsRegressor(),
                    {'n_neighbors': [1, 2, 3, 5, 7, 10, 15, 20, 30, 50],
                     'weights': ['uniform', 'distance'], 'p': [1, 2],
                     'preparation__n_axes': [None, 8, 14]},
                    poids=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
meilleurs = knn.iloc[0]['parametres']
for poids_k in ['uniform', 'distance']:
    sel = knn[knn['parametres'].apply(
        lambda p: p['weights'] == poids_k and p['p'] == meilleurs['p']
        and p['preparation__n_axes'] == meilleurs['preparation__n_axes'])].copy()
    sel['k'] = sel['parametres'].apply(lambda p: p['n_neighbors'])
    sel = sel.sort_values('k')
    ax.plot(sel['k'], sel['RMSE_groupes'], marker='o', label=f'Validation, {poids_k}')
    ax.plot(sel['k'], sel['RMSE_apprentissage'], linestyle='--', alpha=0.6,
            label=f'Apprentissage, {poids_k}')
ax.axhline(reference['RMSE_groupes'].iloc[0], color='grey', linestyle=':', label='Moyenne constante')
ax.set(xscale='log', xlabel='k', ylabel='RMSE par groupes',
       title=f"k plus proches voisins (p={meilleurs['p']}, ACP={meilleurs['preparation__n_axes']})")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'knn_voisins.png', dpi=150); plt.show()

## 7 Comparer avec les arbres du notebook 05

Les plis sont identiques : nous pouvons comparer chaque famille pli par pli. Nous ajoutons la meilleure configuration de chaque famille du notebook 05.

In [ ]:
familles = {'Moyenne constante': reference, 'Régression linéaire': lineaire, 'Ridge': ridge,
            'Lasso': lasso, 'Linéaire après ACP': acp_lineaire, 'k plus proches voisins': knn}
comparaison = pd.concat([t.head(1) for t in familles.values()], ignore_index=True)
comparaison_05 = pd.read_csv(ROOT / 'outputs/05/comparaison_validation.csv')
comparaison_05 = comparaison_05[comparaison_05['modele'] != 'Moyenne constante']
comparaison = pd.concat([comparaison, comparaison_05[comparaison.columns.intersection(comparaison_05.columns)]],
                        ignore_index=True)
comparaison = comparaison.sort_values('RMSE_groupes', kind='stable').reset_index(drop=True)
display(comparaison[['modele', 'parametres', 'RMSE_groupes', 'ecart_type_RMSE',
                     'MAE_groupes', 'R2_groupes', 'RMSE_apprentissage']].round(4))

In [ ]:
par_pli = {}
details = pd.DataFrame(details_plis)
for nom, tableau in familles.items():
    numero = tableau.iloc[0]['configuration']
    par_pli[nom] = details[(details['modele'] == nom) & (details['configuration'] == numero)
                           ].sort_values('pli')['RMSE_groupes'].to_numpy()
for _, ligne in comparaison_05.iterrows():
    sel = details_05[(details_05['modele'] == ligne['modele'])
                     & (details_05['etape'] == ligne['etape'])
                     & (details_05['configuration'] == ligne['configuration'])]
    par_pli[ligne['modele']] = sel.sort_values('pli')['RMSE_groupes'].to_numpy()
par_pli = pd.DataFrame(par_pli, index=pd.Index(range(1, 6), name='pli'))
par_pli = par_pli[comparaison['modele']]
display(par_pli.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
for pli, valeurs in par_pli.iterrows():
    ax.plot(par_pli.columns, valeurs, color='lightgrey', marker='.', zorder=1)
ax.scatter(par_pli.columns, par_pli.mean(), color='black', zorder=2, label='Moyenne des 5 plis')
ax.set(ylabel='RMSE de validation par groupes', title='Comparaison pli par pli')
ax.tick_params(axis='x', rotation=25); ax.legend()
fig.tight_layout(); fig.savefig(OUT / 'comparaison_par_pli.png', dpi=150); plt.show()

## 8 Évaluer sur le test les meilleures configurations

Le test a déjà été consulté pour la forêt dans le notebook 05. Nous y évaluons ici la meilleure configuration de chaque famille de ce notebook, **sans changer aucun choix**. Ces chiffres complètent le tableau du rapport ; la sélection reste fondée sur la validation.

In [ ]:
modeles = {'Moyenne constante': DummyRegressor(strategy='mean'), 'Régression linéaire': LinearRegression(),
           'Ridge': Ridge(), 'Lasso': Lasso(max_iter=50000),
           'Linéaire après ACP': LinearRegression(), 'k plus proches voisins': KNeighborsRegressor()}
y_test_final = calculer_cible(Y_dev, G_dev, Y.loc[test])
np.testing.assert_allclose(y_test_final, donnees.loc[test, 'score_compromis'], atol=1e-12)

resultats_test, predictions_test = {}, {}
for nom, tableau in familles.items():
    parametres = tableau.iloc[0]['parametres']
    params_prep = {k.removeprefix('preparation__'): v for k, v in parametres.items()
                   if k.startswith('preparation__')}
    params_modele = {k: v for k, v in parametres.items() if not k.startswith('preparation__')}
    pipeline = creer_pipeline(X_dev, clone(modeles[nom]).set_params(**params_modele), params_prep)
    options = {} if nom == 'k plus proches voisins' else {'modele__sample_weight': poids_groupes(G_dev)}
    pipeline.fit(X_dev, y_dev_final, **options)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Found unknown categories.*', category=UserWarning)
        predictions_test[nom] = pipeline.predict(X.loc[test])
    resultats_test[nom] = mesurer(y_test_final, predictions_test[nom], groupes.loc[test])

bilan_05 = json.loads((ROOT / 'outputs/05/bilan.json').read_text())
resultats_test['Forêt aléatoire (notebook 05)'] = bilan_05['test_modele_retenu']
resultats_test = pd.DataFrame(resultats_test).T.sort_values('RMSE_groupes')
display(resultats_test.round(4))

## 9 Enregistrer les résultats

In [ ]:
tous = pd.concat(familles.values(), ignore_index=True)
tous.to_csv(OUT / 'toutes_configurations.csv', index=False)
pd.DataFrame(details_plis).to_csv(OUT / 'resultats_par_pli.csv', index=False)
comparaison.to_csv(OUT / 'comparaison_validation.csv', index=False)
par_pli.to_csv(OUT / 'comparaison_par_pli.csv')
resultats_test.to_csv(OUT / 'resultats_test.csv', index_label='modele')
coefficients.to_csv(OUT / 'coefficients_lineaire.csv', index_label='variable', header=['coefficient'])
pd.DataFrame({'groupe': groupes.loc[test], 'y_reel': y_test_final, **predictions_test}).to_csv(
    OUT / 'predictions_test.csv', index_label='row_id')

def convertir(obj):
    if isinstance(obj, np.generic): return obj.item()
    raise TypeError(type(obj).__name__)

bilan = {'meilleures_configurations': {nom: t.iloc[0]['parametres'] for nom, t in familles.items()},
         'RMSE_validation_groupes': {nom: float(t.iloc[0]['RMSE_groupes']) for nom, t in familles.items()},
         'regle_selection': 'plus petite moyenne des 5 RMSE par groupes sur développement',
         'plis': 'GroupKFold(5), identiques au notebook 05',
         'nombre_configurations_modeles': len(tous) - len(reference),
         'test_deja_consulte': True, 'seed': SEED, 'sklearn': sklearn_version,
         'sha256_dataset': hashlib.sha256(source.read_bytes()).hexdigest()}
(OUT / 'bilan.json').write_text(json.dumps(bilan, ensure_ascii=False, indent=2, default=convertir))
print('Résultats enregistrés dans', OUT)

## Conclusion

*À rédiger à partir des tableaux des sections 7 et 8.*